<a href="https://colab.research.google.com/github/Salma53ster/Flash-Chat/blob/master/Download_Clinical_data_Project_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install requests pandas tqdm scipy


### Create a folder for output
All scripts will write outputs into `tcga_tnbc_data/`.


In [1]:
!mkdir -p tcga_tnbc_data


### 1) RNA‑seq expression matrix (CSV)
File: `01_download_rnaseq_matrix.py`


In [2]:
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import gzip # Import gzip module

GDC_API = "https://api.gdc.cancer.gov"
PROJECT = "TCGA-BRCA"
TOKEN_PATH = "gdc_token.txt"
OUTPUT_DIR = Path("tcga_tnbc_data")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS_WITH_TOKEN = {}
try:
    with open(TOKEN_PATH, "r") as f:
        token = f.read().strip()
    HEADERS_WITH_TOKEN = {"X-GDC-TOKEN": token}
except FileNotFoundError:
    pass


def post_gdc(endpoint, filters, fields=None, size=200):
    url = f"{GDC_API}/{endpoint}"
    payload = {"filters": filters, "format": "JSON", "size": size}
    if fields:
        payload["fields"] = fields

    all_hits = []
    from_val = 0

    while True:
        params = {"from": from_val}
        resp = requests.post(url, params=params, json=payload, headers=HEADERS_WITH_TOKEN)
        resp.raise_for_status()
        data = resp.json()
        hits = data["data"]["hits"]
        all_hits.extend(hits)

        pagination = data["data"]["pagination"]
        # Changed 'per_page' to 'size' based on common GDC API pagination structure
        if pagination["page"] * pagination["size"] >= pagination["total"]:
            break
        from_val += pagination["size"]

    return all_hits


def get_rnaseq_files(workflow_type="STAR - Counts"):
    filters = {
        "op": "and",
        "content": [
            {"op": "=", "content": {"field": "cases.project.project_id", "value": PROJECT}},
            {"op": "=", "content": {"field": "files.data_category", "value": "Transcriptome Profiling"}},
            {"op": "=", "content": {"field": "files.data_type", "value": "Gene Expression Quantification"}},
            {"op": "=", "content": {"field": "files.experimental_strategy", "value": "RNA-Seq"}},
            {"op": "=", "content": {"field": "files.analysis.workflow_type", "value": workflow_type}},
            {"op": "in", "content": {"field": "cases.samples.sample_type", "value": ["Primary Tumor", "Solid Tissue Normal"]}},
        ]
    }

    fields = ",".join([
        "file_id",
        "file_name",
        "analysis.workflow_type",
        "cases.project.project_id",
        "cases.samples.sample_type",
        "cases.submitter_id"
    ])

    hits = post_gdc("files", filters, fields=fields)
    print(f"Found {len(hits)} RNA-seq files for workflow: {workflow_type}")
    return hits


def download_file(file_id, out_path):
    url = f"{GDC_API}/data/{file_id}"
    with requests.get(url, headers=HEADERS_WITH_TOKEN, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)


def build_and_save_rnaseq_matrix(file_info, output_csv):
    all_dfs = []
    sample_names = []

    for info in tqdm(file_info, desc="Downloading RNA-seq files"):
        fid = info["file_id"]
        tmp_path = OUTPUT_DIR / f"{fid}.tsv.gz" # Still download with .gz extension as it might be gzipped
        download_file(fid, tmp_path)

        df = None
        try:
            # Check if file is actually gzipped by reading magic bytes
            is_gzipped = False
            with open(tmp_path, 'rb') as f:
                magic_bytes = f.read(2)
                if magic_bytes == b'\x1f\x8b': # Gzip magic bytes
                    is_gzipped = True

            if is_gzipped:
                df = pd.read_csv(tmp_path, sep="\t", compression='gzip', header=None, comment='#')
            else:
                print(f"Warning: File {tmp_path} is not gzipped despite extension. Attempting to read as plain TSV with comments.")
                df = pd.read_csv(tmp_path, sep="\t", compression=None, header=None, comment='#') # Explicitly no compression

        except Exception as e:
            print(f"Error: Could not read file {tmp_path}: {e}. Skipping this file.")
            tmp_path.unlink(missing_ok=True) # Clean up partial file
            continue # Skip this file and continue with the next one

        if df is None or df.empty:
            print(f"Warning: Skipping empty or unreadable file {tmp_path}.")
            tmp_path.unlink(missing_ok=True) # Clean up partial file
            continue

        # Ensure the DataFrame has at least two columns before proceeding
        if df.shape[1] < 2:
            print(f"Warning: File {tmp_path} does not have enough columns. Skipping.")
            tmp_path.unlink(missing_ok=True) # Clean up partial file
            continue

        df = df.iloc[:, :2]
        df.columns = ["gene_id", fid]

        # Drop duplicate gene_ids, keeping the first occurrence
        df = df.drop_duplicates(subset=['gene_id'], keep='first')
        df = df.set_index("gene_id")

        all_dfs.append(df)

        case_sub = info.get("cases", [{}])[0].get("submitter_id", fid)
        sample_type = info.get("cases", [{}])[0].get("samples", [{}])[0].get("sample_type", "UNK")
        sample_name = f"{case_sub}_{sample_type}"
        sample_names.append(sample_name)

        # Optionally delete tmp file:
        tmp_path.unlink(missing_ok=True) # Delete temp file after processing

    if not all_dfs:
        print("No valid RNA-seq files were processed.")
        return pd.DataFrame()

    merged = pd.concat(all_dfs, axis=1)
    merged.columns = sample_names
    merged.to_csv(output_csv)
    print(f"RNA-seq matrix saved to: {output_csv}")
    return merged


# __name__ == "__main__" block
# Choose workflow: "STAR - Counts", "HTSeq - Counts", "STAR - FPKM", "STAR - TPM"
rna_files = get_rnaseq_files(workflow_type="STAR - Counts")

if not rna_files:
    print("No RNA-seq files found. Try another workflow_type.")
else:
    build_and_save_rnaseq_matrix(
        rna_files,
        OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"
    )

Found 1224 RNA-seq files for workflow: STAR - Counts


RNA-seq matrix saved to: tcga_tnbc_data/TCGA_BRCA_RNAseq_matrix.csv


**Output:**
`tcga_tnbc_data/TCGA_BRCA_RNAseq_matrix.csv` (gene × sample, counts/FPKM/TPM depending on workflow).


### Retrying RNA-seq expression matrix download and processing

Based on the previous truncated output and the subsequent `RNA-seq matrix not found` error, the RNA-seq data download and processing for `TCGA_BRCA_RNAseq_matrix.csv` did not complete successfully. I will re-run the cell responsible for this task to ensure all files are downloaded and the matrix is properly constructed.

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import gzip
import os
import shutil # For removing temporary directories

GDC_API = "https://api.gdc.cancer.gov"
PROJECT = "TCGA-BRCA"
TOKEN_PATH = "gdc_token.txt"
OUTPUT_DIR = Path("tcga_tnbc_data")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS_WITH_TOKEN = {}
try:
    with open(TOKEN_PATH, "r") as f:
        token = f.read().strip()
    HEADERS_WITH_TOKEN = {"X-GDC-TOKEN": token}
except FileNotFoundError:
    pass


def post_gdc(endpoint, filters, fields=None, size=200):
    url = f"{GDC_API}/{endpoint}"
    payload = {"filters": filters, "format": "JSON", "size": size}
    if fields:
        payload["fields"] = fields

    all_hits = []
    from_val = 0

    while True:
        params = {"from": from_val}
        resp = requests.post(url, params=params, json=payload, headers=HEADERS_WITH_TOKEN)
        resp.raise_for_status()
        data = resp.json()
        hits = data["data"]["hits"]
        all_hits.extend(hits)

        pagination = data["data"]["pagination"]
        if pagination["page"] * pagination["size"] >= pagination["total"]:
            break
        from_val += pagination["size"]

    return all_hits


def get_rnaseq_files(workflow_type="STAR - Counts"):
    filters = {
        "op": "and",
        "content": [
            {"op": "=", "content": {"field": "cases.project.project_id", "value": PROJECT}},
            {"op": "=", "content": {"field": "files.data_category", "value": "Transcriptome Profiling"}},
            {"op": "=", "content": {"field": "files.data_type", "value": "Gene Expression Quantification"}},
            {"op": "=", "content": {"field": "files.experimental_strategy", "value": "RNA-Seq"}},
            {"op": "=", "content": {"field": "files.analysis.workflow_type", "value": workflow_type}},
            {"op": "in", "content": {"field": "cases.samples.sample_type", "value": ["Primary Tumor", "Solid Tissue Normal"]}},
        ]
    }

    fields = ",".join([
        "file_id",
        "file_name",
        "analysis.workflow_type",
        "cases.project.project_id",
        "cases.samples.sample_type",
        "cases.submitter_id"
    ])

    hits = post_gdc("files", filters, fields=fields)
    print(f"Found {len(hits)} RNA-seq files for workflow: {workflow_type}")
    return hits


def download_file(file_id, out_path):
    url = f"{GDC_API}/data/{file_id}"
    with requests.get(url, headers=HEADERS_WITH_TOKEN, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)


def build_and_save_rnaseq_matrix(file_info, output_csv):
    # 1. Initial check for final output
    if output_csv.exists() and os.path.getsize(output_csv) > 0:
        print(f"Final RNA-seq matrix '{output_csv}' already exists and is not empty. Skipping processing.")
        return pd.read_csv(output_csv, index_col=0) # Return the existing dataframe

    temp_dir = OUTPUT_DIR / "rnaseq_temp_files"
    temp_dir.mkdir(exist_ok=True)

    # Get a set of file_ids that have already been processed and saved as temp CSVs
    existing_temp_fids = {f.stem for f in temp_dir.glob('*.csv')}

    # Prepare list of files that still need to be processed
    files_to_process_now = []
    total_expected_files = len(file_info)
    currently_processed_in_temp = len(existing_temp_fids)

    # Separate files already processed from those to be processed
    for info in file_info:
        fid = info["file_id"]
        if fid not in existing_temp_fids:
            files_to_process_now.append(info)

    print(f"Resuming processing: {currently_processed_in_temp} files already processed, {len(files_to_process_now)} remaining to process.")

    # Process remaining files
    for info in tqdm(files_to_process_now, desc="Processing RNA-seq files"):
        fid = info["file_id"]
        case_sub = info.get("cases", [{}])[0].get("submitter_id", fid)
        sample_type = info.get("cases", [{}])[0].get("samples", [{}])[0].get("sample_type", "UNK")
        sample_name = f"{case_sub}_{sample_type}"

        temp_processed_path = temp_dir / f"{fid}.csv"
        tmp_download_path = temp_dir / f"{fid}.tsv.gz"

        try:
            download_file(fid, tmp_download_path)

            df = None
            is_gzipped = False
            with open(tmp_download_path, 'rb') as f:
                magic_bytes = f.read(2)
                if magic_bytes == b'\x1f\x8b':
                    is_gzipped = True

            if is_gzipped:
                df = pd.read_csv(tmp_download_path, sep="\t", compression='gzip', header=None, comment='#')
            else:
                print(f"Warning: File {tmp_download_path} is not gzipped despite extension. Attempting to read as plain TSV with comments.")
                df = pd.read_csv(tmp_download_path, sep="\t", compression=None, header=None, comment='#')

        except requests.exceptions.RequestException as e:
            print(f"Network error downloading file {fid}: {e}. Skipping this file.")
            tmp_download_path.unlink(missing_ok=True)
            continue
        except Exception as e:
            print(f"Error: Could not read or process file {tmp_download_path}: {e}. Skipping this file.")
            tmp_download_path.unlink(missing_ok=True)
            continue

        if df is None or df.empty:
            print(f"Warning: Skipping empty or unreadable file {tmp_download_path}.")
            tmp_download_path.unlink(missing_ok=True)
            continue

        if df.shape[1] < 2:
            print(f"Warning: File {tmp_download_path} does not have enough columns. Skipping.")
            tmp_download_path.unlink(missing_ok=True)
            continue

        df = df.iloc[:, :2]
        df.columns = ["gene_id", sample_name]
        df = df.drop_duplicates(subset=['gene_id'], keep='first')
        df = df.set_index("gene_id")

        df.to_csv(temp_processed_path)
        tmp_download_path.unlink(missing_ok=True) # Delete temp downloaded file after processing

    # After attempting to process all files, check if all expected temp CSVs are present
    final_processed_count = len(list(temp_dir.glob('*.csv')))
    if final_processed_count < total_expected_files:
        print(f"Error: Only {final_processed_count} of {total_expected_files} files were successfully processed. "
              "Final RNA-seq matrix will be incomplete or not generated. Please re-run the cell to continue.")
        return pd.DataFrame() # Return empty DataFrame to indicate failure/incompletion

    # Second pass: read all temporary CSVs and concatenate
    print("All individual RNA-seq files processed. Concatenating into final matrix...")
    all_dfs_to_concat = []
    temp_csv_files = list(temp_dir.glob('*.csv'))

    for f_path in tqdm(temp_csv_files, desc="Reading temporary CSVs and concatenating"):
        try:
            df_temp = pd.read_csv(f_path, index_col=0)
            all_dfs_to_concat.append(df_temp)
        except Exception as e:
            print(f"Error reading temp file {f_path} during concatenation: {e}. Skipping.")

    if not all_dfs_to_concat:
        print("No valid dataframes to concatenate for the final matrix.")
        return pd.DataFrame()

    merged = pd.concat(all_dfs_to_concat, axis=1)
    merged.to_csv(output_csv)
    print(f"RNA-seq matrix saved to: {output_csv}")

    # Final verification and cleanup
    if output_csv.exists() and os.path.getsize(output_csv) > 0:
        print(f"Successfully created final matrix. Cleaning up temporary directory: {temp_dir}")
        shutil.rmtree(temp_dir, ignore_errors=True)
        return merged
    else:
        print(f"Error: Final matrix '{output_csv}' could not be verified after saving. "
              "Temporary files were not removed to allow debugging.")
        return pd.DataFrame()


# __name__ == "__main__" block
# Choose workflow: "STAR - Counts", "HTSeq - Counts", "STAR - FPKM", "STAR - TPM"
rna_files = get_rnaseq_files(workflow_type="STAR - Counts")

if not rna_files:
    print("No RNA-seq files found. Try another workflow_type.")
else:
    build_and_save_rnaseq_matrix(
        rna_files,
        OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"
    )

Found 1224 RNA-seq files for workflow: STAR - Counts
Resuming processing: 1224 files already processed, 0 remaining to process.


Processing RNA-seq files: 0it [00:00, ?it/s]


All individual RNA-seq files processed. Concatenating into final matrix...


Reading temporary CSVs and concatenating:  59%|█████▉    | 725/1224 [01:32<01:16,  6.51it/s]

In [1]:
import os
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")

if OUTPUT_DIR.exists() and OUTPUT_DIR.is_dir():
    print(f"Contents of '{OUTPUT_DIR}':")
    for item in os.listdir(OUTPUT_DIR):
        print(f"- {item}")
else:
    print(f"Directory '{OUTPUT_DIR}' does not exist.")

Contents of 'tcga_tnbc_data':
- 12b68e56-a83f-4b65-bb66-59e8522acbc9.tsv.gz


### 2) miRNA expression matrix (CSV)
File: `02_download_mirna_matrix.py`


In [3]:
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import gzip # Import gzip module

GDC_API = "https://api.gdc.cancer.gov"
PROJECT = "TCGA-BRCA"
TOKEN_PATH = "gdc_token.txt"
OUTPUT_DIR = Path("tcga_tnbc_data")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS_WITH_TOKEN = {}
try:
    with open(TOKEN_PATH, "r") as f:
        token = f.read().strip()
    HEADERS_WITH_TOKEN = {"X-GDC-TOKEN": token}
except FileNotFoundError:
    pass


def post_gdc(endpoint, filters, fields=None, size=200):
    url = f"{GDC_API}/{endpoint}"
    payload = {"filters": filters, "format": "JSON", "size": size}
    if fields:
        payload["fields"] = fields

    all_hits = []
    from_val = 0

    while True:
        params = {"from": from_val}
        resp = requests.post(url, params=params, json=payload, headers=HEADERS_WITH_TOKEN)
        resp.raise_for_status()
        data = resp.json()
        hits = data["data"]["hits"]
        all_hits.extend(hits)

        pagination = data["data"]["pagination"]
        # Changed 'per_page' to 'size' based on common GDC API pagination structure
        if pagination["page"] * pagination["size"] >= pagination["total"]:
            break
        from_val += pagination["size"]

    return all_hits


def get_mirna_files():
    filters = {
        "op": "and",
        "content": [
            {"op": "=", "content": {"field": "cases.project.project_id", "value": PROJECT}},
            {"op": "=", "content": {"field": "files.data_category", "value": "Transcriptome Profiling"}},
            {"op": "=", "content": {"field": "files.data_type", "value": "miRNA Expression Quantification"}},
            {"op": "=", "content": {"field": "files.experimental_strategy", "value": "miRNA-Seq"}},
            {"op": "in", "content": {"field": "cases.samples.sample_type", "value": ["Primary Tumor", "Solid Tissue Normal"]}},
        ]
    }

    fields = ",".join([
        "file_id",
        "file_name",
        "cases.project.project_id",
        "cases.samples.sample_type",
        "cases.submitter_id"
    ])

    hits = post_gdc("files", filters, fields=fields)
    print(f"Found {len(hits)} miRNA files.")
    return hits


def download_file(file_id, out_path):
    url = f"{GDC_API}/data/{file_id}"
    with requests.get(url, headers=HEADERS_WITH_TOKEN, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)


def build_and_save_mirna_matrix(file_info, output_csv):
    all_dfs = []
    sample_names = []

    for info in tqdm(file_info, desc="Downloading miRNA files"):
        fid = info["file_id"]
        tmp_path = OUTPUT_DIR / f"{fid}_mirna.tsv.gz"
        download_file(fid, tmp_path)

        df = None
        try:
            # Check if file is actually gzipped by reading magic bytes
            is_gzipped = False
            with open(tmp_path, 'rb') as f:
                magic_bytes = f.read(2)
                if magic_bytes == b'\x1f\x8b': # Gzip magic bytes
                    is_gzipped = True

            if is_gzipped:
                df = pd.read_csv(tmp_path, sep="\t", compression='gzip', comment='#') # miRNA files often have headers
            else:
                print(f"Warning: File {tmp_path} is not gzipped despite extension. Attempting to read as plain TSV with comments.")
                df = pd.read_csv(tmp_path, sep="\t", compression=None, comment='#') # Explicitly no compression

        except Exception as e:
            print(f"Error: An unexpected error occurred while reading file {tmp_path}: {e}. Skipping this file.")
            tmp_path.unlink(missing_ok=True)
            continue

        if df is None or df.empty:
            print(f"Warning: Skipping empty or unreadable file {tmp_path}.")
            tmp_path.unlink(missing_ok=True)
            continue

        if "reads_per_million_miRNA_mapped" in df.columns:
            value_col = "reads_per_million_miRNA_mapped"
        else:
            # Fallback to the second column if the specific column name is not found
            # Assuming miRNA_ID is first column and value is second
            if df.shape[1] > 1:
                value_col = df.columns[1]
            else:
                print(f"Warning: File {tmp_path} does not have enough columns for miRNA expression. Skipping.")
                tmp_path.unlink(missing_ok=True)
                continue

        sub = df[["miRNA_ID", value_col]].copy()
        sub = sub.rename(columns={"miRNA_ID": "miRNA_id", value_col: fid})
        sub = sub.set_index("miRNA_id")

        all_dfs.append(sub)

        case_sub = info.get("cases", [{}])[0].get("submitter_id", fid)
        sample_type = info.get("cases", [{}])[0].get("samples", [{}])[0].get("sample_type", "UNK")
        sample_name = f"{case_sub}_{sample_type}"
        sample_names.append(sample_name)

        tmp_path.unlink(missing_ok=True) # Delete temp file after processing

    if not all_dfs:
        print("No valid miRNA files were processed.")
        return pd.DataFrame()

    merged = pd.concat(all_dfs, axis=1)
    merged.columns = sample_names
    merged.to_csv(output_csv)
    print(f"miRNA matrix saved to: {output_csv}")
    return merged


# __name__ == "__main__" block
mirna_files = get_mirna_files()
if not mirna_files:
    print("No miRNA files found.")
else:
    build_and_save_mirna_matrix(
        mirna_files,
        OUTPUT_DIR / "TCGA_BRCA_miRNA_matrix.csv"
    )

Found 1200 miRNA files.


miRNA matrix saved to: tcga_tnbc_data/TCGA_BRCA_miRNA_matrix.csv


In [4]:
import os
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")

if OUTPUT_DIR.exists() and OUTPUT_DIR.is_dir():
    print(f"Contents of '{OUTPUT_DIR}':")
    for item in os.listdir(OUTPUT_DIR):
        print(f"- {item}")
else:
    print(f"Directory '{OUTPUT_DIR}' does not exist.")

Contents of 'tcga_tnbc_data':
- TCGA_BRCA_RNAseq_matrix.csv
- TCGA_BRCA_miRNA_matrix.csv


### Running Differential Gene Expression Analysis

Now that the RNA-seq expression matrix and clinical data (including TNBC status) are available, we can proceed with the differential gene expression analysis (DEG). This step will identify genes that are significantly differentially expressed between TNBC and non-TNBC samples.

In [6]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
from statsmodels.stats.multitest import multipletests

OUTPUT_DIR = Path("tcga_tnbc_data")


def simple_deg_analysis(
    rna_matrix_path,
    clinical_path,
    output_deg_path,
    tnbc_col="TNBC",
    min_samples_per_group=5,
):
    expr = pd.read_csv(rna_matrix_path, index_col=0)
    clin = pd.read_csv(clinical_path)

    # Map sample names in expression to submitter_id / case barcode
    # Expression columns are like: TCGA-XX-XXXX_ Primary Tumor
    # We'll extract the TCGA barcode part before the first underscore.
    def extract_barcode(col):
        return str(col).split("_")[0]

    sample_barcodes = [extract_barcode(c) for c in expr.columns]
    expr_tmp = expr.copy()
    expr_tmp.columns = sample_barcodes

    # Merge clinical info
    clin = clin.set_index("submitter_id")
    common = list(set(expr_tmp.columns) & set(clin.index))
    print(f"Common samples between expression and clinical: {len(common)}")

    expr_common = expr_tmp[common]
    clin_common = clin.loc[common]

    if tnbc_col not in clin_common.columns:
        raise ValueError(f"Column '{tnbc_col}' not found in clinical file.")

    group = clin_common[tnbc_col].astype(int)
    tnbc_samples = group[group == 1].index.tolist()
    other_samples = group[group == 0].index.tolist()

    if len(tnbc_samples) < min_samples_per_group or len(other_samples) < min_samples_per_group:
        raise ValueError("Not enough samples in one of the groups.")

    # Simple t-test per gene
    genes = expr_common.index.tolist()
    logfc_list = []
    pvals = []

    for g in genes:
        x = expr_common.loc[g, tnbc_samples].astype(float).dropna()
        y = expr_common.loc[g, other_samples].astype(float).dropna()

        if len(x) < 2 or len(y) < 2:
            logfc_list.append(np.nan)
            pvals.append(1.0)
            continue

        m1 = np.log2(x + 1).mean()
        m2 = np.log2(y + 1).mean()
        logfc = m1 - m2

        # t-test on log2(x+1)
        # The original code has an error here, stats.t_ind_test does not exist.
        # It should be stats.ttest_ind
        t_stat, p_val = stats.ttest_ind(np.log2(x + 1), np.log2(y + 1), equal_var=False) # Assuming unequal variances
        logfc_list.append(logfc)
        pvals.append(p_val)

    deg_df = pd.DataFrame({
        "gene": genes,
        "log2FC": logfc_list,
        "pval": pvals,
    })

    # FDR correction
    _, adj_pvals, _, _ = multipletests(deg_df["pval"].fillna(1.0), method="fdr_bh")
    deg_df["adj_pval"] = adj_pvals

    # Sort by adj_pval
    deg_df = deg_df.sort_values("adj_pval").reset_index(drop=True)

    deg_df.to_csv(output_deg_path, index=False)
    print(f"DEG list saved to: {output_deg_path}")
    return deg_df


if __name__ == "__main__":
    rna_matrix = OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"
    clinical = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"
    output_deg = OUTPUT_DIR / "TNBC_DEG_list_for_GO_KEGG.csv"

    if not rna_matrix.exists():
        print("RNA-seq matrix not found. Run 01_download_rnaseq_matrix.py first.")
    elif not clinical.exists():
        print("Clinical file not found. Run 03_download_clinical_TNBC.py first.")
    else:
        deg = simple_deg_analysis(rna_matrix, clinical, output_deg)


Clinical file not found. Run 03_download_clinical_TNBC.py first.


**Output:**
`tcga_tnbc_data/TCGA_BRCA_miRNA_matrix.csv` (miRNA × sample, RPM).


### 3) Clinical data with TNBC status (CSV)
This script:
- Pulls clinical data from GDC (stage, survival, etc.)
- Pulls receptor status (ER/PR/HER2) from cBioPortal’s public TCGA‑BRCA study
- Merges them and defines TNBC = ER‑, PR‑, HER2‑.

File: `03_download_clinical_TNBC.py`


In [15]:
import requests
import pandas as pd
from pathlib import Path
import time
import os # Import os module to check file size

GDC_API = "https://api.gdc.cancer.gov"
PROJECT = "TCGA-BRCA"
TOKEN_PATH = "gdc_token.txt"
OUTPUT_DIR = Path("tcga_tnbc_data")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS_WITH_TOKEN = {}
try:
    with open(TOKEN_PATH, "r") as f:
        token = f.read().strip()
    HEADERS_WITH_TOKEN = {"X-GDC-TOKEN": token}
except FileNotFoundError:
    pass


def post_gdc(endpoint, filters, fields=None, size=200):
    url = f"{GDC_API}/{endpoint}"
    payload = {"filters": filters, "format": "JSON", "size": size}
    if fields:
        payload["fields"] = fields

    all_hits = []
    from_val = 0

    while True:
        params = {"from": from_val}
        resp = requests.post(url, params=params, json=payload, headers=HEADERS_WITH_TOKEN)
        resp.raise_for_status()
        data = resp.json()
        hits = data["data"]["hits"]
        all_hits.extend(hits)

        pagination = data["data"]["pagination"]
        # Changed 'per_page' to 'size' based on common GDC API pagination structure
        if pagination["page"] * pagination["size"] >= pagination["total"]:
            break
        from_val += pagination["size"]

    return all_hits


def get_gdc_clinical():
    filters = {
        "op": "=",
        "content": {"field": "cases.project.project_id", "value": PROJECT},
    }

    fields = ",".join([
        "case_id",
        "submitter_id",
        "demographic.gender",
        "demographic.race",
        "demographic.ethnicity",
        "diagnoses.age_at_diagnosis",
        "diagnoses.tumor_stage",
        "diagnoses.tumor_grade",
        "diagnoses.primary_diagnosis",
        "diagnoses.days_to_death",
        "diagnoses.days_to_last_follow_up",
        "diagnoses.vital_status",
        "diagnoses.progression_or_recurrence",
        "diagnoses.days_to_progression_or_recurrence",
        "diagnoses.er_status_by_ihc", # Added GDC ER status
        "diagnoses.pr_status_by_ihc", # Added GDC PR status
        "diagnoses.her2_status_by_ihc", # Added GDC HER2 status
        "samples.sample_type",
        "project.project_id",
    ])

    hits = post_gdc("cases", filters, fields=fields)

    rows = []
    for h in hits:
        case_id = h.get("case_id")
        submitter_id = h.get("submitter_id")
        demo = h.get("demographic", {})
        diag_list = h.get("diagnoses", [])
        samples_list = h.get("samples", [])

        diag = diag_list[0] if diag_list else {}
        sample = samples_list[0] if samples_list else {}

        row = {
            "case_id": case_id,
            "submitter_id": submitter_id,
            "gender": demo.get("gender"),
            "race": demo.get("race"),
            "ethnicity": demo.get("ethnicity"),
            "age_at_diagnosis": diag.get("age_at_diagnosis"),
            "tumor_stage": diag.get("tumor_stage"),
            "tumor_grade": diag.get("tumor_grade"),
            "primary_diagnosis": diag.get("primary_diagnosis"),
            "days_to_death": diag.get("days_to_death"),
            "days_to_last_follow_up": diag.get("days_to_last_follow_up"),
            "vital_status": diag.get("vital_status"),
            "progression_or_recurrence": diag.get("progression_or_recurrence"),
            "days_to_progression_or_recurrence": diag.get("days_to_progression_or_recurrence"),
            "er_status_gdc": diag.get("er_status_by_ihc"), # Extract GDC ER status
            "pr_status_gdc": diag.get("pr_status_by_ihc"), # Extract GDC PR status
            "her2_status_gdc": diag.get("her2_status_by_ihc"), # Extract GDC HER2 status
            "sample_type": sample.get("sample_type"),
            "project_id": h.get("project", {}).get("project_id"),
        }
        rows.append(row)

    clin_gdc = pd.DataFrame(rows)
    clin_gdc.to_csv(OUTPUT_DIR / "TCGA_BRCA_clinical_GDC.csv", index=False)
    print(f"GDC clinical saved to: {OUTPUT_DIR / 'TCGA_BRCA_clinical_GDC.csv'}")
    return clin_gdc


def get_cbioportal_clinical(study_id, out_file_prefix="TCGA_BRCA"):
    """
    Fetch clinical data from cBioPortal (public API) for a given study.
    We'll get patient-level attributes including ER/PR/HER2 if available.
    """
    CBIO_API = "https://www.cbioportal.org/api"

    # 1. Get all patient IDs for the study
    url_patients = f"{CBIO_API}/studies/{study_id}/patients"
    params_patients = {"pageSize": 10000} # Get all patients
    resp_patients = requests.get(url_patients, params=params_patients)
    resp_patients.raise_for_status()
    patients_summary = resp_patients.json()
    patient_ids = [p['patientId'] for p in patients_summary]

    if not patient_ids:
        print(f"No patient data found for {study_id} from cBioPortal.")
        return pd.DataFrame()

    # 2. Get all clinical attribute metadata for the study
    url_clinical_attributes = f"{CBIO_API}/studies/{study_id}/clinical-attributes"
    params_clinical_attributes = {"pageSize": 10000}
    resp_clinical_attributes = requests.get(url_clinical_attributes, params=params_clinical_attributes)
    resp_clinical_attributes.raise_for_status()
    clinical_attribute_meta = resp_clinical_attributes.json()

    # Map clinical attribute IDs to display names for column headers
    attr_id_to_name = {attr['clinicalAttributeId']: attr['displayName'] for attr in clinical_attribute_meta}

    print(f"Available clinical attributes for {study_id} (ID: Display Name):")
    for attr in clinical_attribute_meta:
        print(f"  {attr['clinicalAttributeId']}: {attr['displayName']}")

    # Explicitly define clinical attribute IDs for ER, PR, HER2 to request
    # These are specific for TCGA_BRCA, will need to be adapted for METABRIC
    relevant_attr_ids = ["ER_STATUS_BY_IHC", "PR_STATUS_BY_IHC", "HER2_IHC_STATUS"]

    # Filter for only those attributes that actually exist in the study metadata
    available_relevant_attr_ids = [aid for aid in relevant_attr_ids if aid in attr_id_to_name]
    clinical_attribute_ids_str = ",".join(available_relevant_attr_ids)

    if not clinical_attribute_ids_str:
        print(f"Warning: No known ER/PR/HER2 attributes found for {study_id}. Proceeding with all available attributes.")
        clinical_attribute_ids_str = ",".join(attr_id_to_name.keys())

    # 3. Fetch clinical data for all patients and selected attributes using GET
    url_clinical_data = f"{CBIO_API}/studies/{study_id}/clinical-data"

    all_raw_clinical_data = []
    patient_id_chunks = [patient_ids[i:i + 100] for i in range(0, len(patient_ids), 100)] # Chunk patient IDs

    for i, patient_chunk in enumerate(patient_id_chunks):
        print(f"Fetching clinical data for patient chunk {i+1}/{len(patient_id_chunks)} for {study_id}...")
        # Prepare query parameters for GET request for the current chunk
        params_clinical_data = {
            "patientIds": ",".join(patient_chunk),
            "clinicalAttributeIds": clinical_attribute_ids_str # Re-enabled to explicitly request ER/PR/HER2
        }

        # Add retry logic for the GET request
        MAX_RETRIES = 5
        for attempt in range(MAX_RETRIES):
            try:
                resp_clinical_data = requests.get(url_clinical_data, params=params_clinical_data, timeout=60)
                resp_clinical_data.raise_for_status()
                all_raw_clinical_data.extend(resp_clinical_data.json())
                break # Break out of the retry loop if successful
            except requests.exceptions.RequestException as e:
                print(f"RequestException encountered for chunk {i+1} of {study_id} (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2 ** attempt) # Exponential back-off
                else:
                    raise # Re-raise the last exception if all retries fail
        else:
            # This block is executed if the loop completes without a 'break'
            print(f"Failed to retrieve clinical data for chunk {i+1} of {study_id} after {MAX_RETRIES} attempts. Skipping this chunk.")

    # DEBUG: Print unique clinicalAttributeIds found in the raw data
    if all_raw_clinical_data:
        raw_attribute_ids = {item.get('clinicalAttributeId') for item in all_raw_clinical_data}
        print(f"Unique clinicalAttributeIds found in raw data for {study_id} (before mapping/pivot):")
        print(list(raw_attribute_ids))
    else:
        print(f"No raw clinical data returned from cBioPortal for {study_id} for the requested attributes.")

    # 4. Process raw clinical data into a DataFrame
    # The data comes in a 'long' format, need to pivot it to 'wide'
    clinical_rows = []
    for item in all_raw_clinical_data:
        clinical_rows.append({
            "patient_id": item["patientId"],
            "attribute_name": attr_id_to_name.get(item["clinicalAttributeId"], item["clinicalAttributeId"]),
            "attribute_value": item["value"]
        })

    df_clinical_long = pd.DataFrame(clinical_rows)

    if df_clinical_long.empty:
        print(f"No detailed clinical attributes found for {study_id} from cBioPortal.")
        return pd.DataFrame()

    print(f"Unique attribute names found in raw clinical data for {study_id} (before pivot):")
    print(df_clinical_long['attribute_name'].unique().tolist())

    # Handle potential duplicate (patient_id, attribute_name) combinations
    # Keep the first occurrence if duplicates exist to allow pivoting.
    df_clinical_long = df_clinical_long.drop_duplicates(subset=['patient_id', 'attribute_name'], keep='first')

    # Pivot the table to get attributes as columns
    clin_cbio = df_clinical_long.pivot(index='patient_id', columns='attribute_name', values='attribute_value')
    clin_cbio = clin_cbio.reset_index()
    clin_cbio.columns.name = None # Remove the name of the columns index

    print(f"Columns found in cBioPortal clinical data for {study_id}:")
    print(clin_cbio.columns.tolist())

    # Add a dummy sample_id for merging purposes if not naturally present
    if 'sample_id' not in clin_cbio.columns:
        clin_cbio['sample_id'] = None # Or extract from patient_id if a pattern exists

    out_path_cbio = OUTPUT_DIR / f"{out_file_prefix}_clinical_cBio.csv"
    clin_cbio.to_csv(out_path_cbio, index=False)
    print(f"cBioPortal clinical for {study_id} saved to: {out_path_cbio}")
    return clin_cbio


def merge_and_define_TNBC():
    clin_gdc = pd.read_csv(OUTPUT_DIR / "TCGA_BRCA_clinical_GDC.csv")
    clin_cbio = pd.read_csv(OUTPUT_DIR / "TCGA_BRCA_clinical_cBio.csv")

    # Map cBioPortal patient_id (e.g. TCGA-XX-XXXX) to GDC submitter_id
    # GDC submitter_id is usually the same as patient barcode.
    # We'll merge on that.
    clin_cbio = clin_cbio.rename(columns={"patient_id": "submitter_id"})

    merged = clin_gdc.merge(
        clin_cbio,
        on="submitter_id",
        how="left",
        suffixes=('_GDC', '_cBio'),
    )

    # Define ER/PR/HER2 status from GDC or cBioPortal columns
    er_col = None
    pr_col = None
    her2_col = None

    # Prioritize GDC columns first, then cBioPortal display names from metadata
    possible_er_cols = ["er_status_gdc", "ER Status By IHC", "ER_STATUS", "ER_STATUS_BY_IMMUNOHISTOCHEMISTRY", "ESTROGEN_RECEPTOR_STATUS"]
    possible_pr_cols = ["pr_status_gdc", "PR status by ihc", "PR Status By IHC", "PROGESTERONE_RECEPTOR_STATUS", "PROGESTERONE_RECEPTOR_STATUS_BY_IMMUNOHISTOCHEMISTRY"]
    possible_her2_cols = ["her2_status_gdc", "HER2 IHC Status", "HER2 Status By IHC", "HER2_STATUS", "HER2_STATUS_BY_IMMUNOHISTOCHEMISTRY"]

    for p_col in possible_er_cols:
        if p_col in merged.columns:
            er_col = p_col
            break
    for p_col in possible_pr_cols:
        if p_col in merged.columns:
            pr_col = p_col
            break
    for p_col in possible_her2_cols:
        if p_col in merged.columns:
            her2_col = p_col
            break

    print("Detected receptor columns:")
    print("ER:", er_col, "PR:", pr_col, "HER2:", her2_col)

    def is_negative(val):
        if pd.isna(val):
            return False
        v = str(val).strip().lower()
        # Also consider 'equivocal' as non-negative for TNBC definition, or handle explicitly
        return v in ["negative", "neg", "0", "no"]

    def is_positive(val):
        if pd.isna(val):
            return False
        v = str(val).strip().lower()
        return v in ["positive", "pos", "1", "yes"]

    if er_col and pr_col and her2_col:
        merged["ER_neg"] = merged[er_col].apply(is_negative)
        merged["PR_neg"] = merged[pr_col].apply(is_negative)
        merged["HER2_neg"] = merged[her2_col].apply(is_negative)

        merged["TNBC"] = (merged["ER_neg"] & merged["PR_neg"] & merged["HER2_neg"]).astype(int)
    else:
        print("Could not detect all ER/PR/HER2 columns; TNBC column will not be created.")
        merged["TNBC"] = pd.NA

    # Survival variables
    # Try to create OS_time (months) and OS_event (0/1)
    if "OS Months" in merged.columns and "OS Status" in merged.columns:
        merged["OS_months"] = merged["OS Months"]
        merged["OS_event"] = (
            merged["OS Status"]
            .astype(str)
            .str.lower()
            .map(lambda x: 1 if "deceased" in x else 0)
        )

    # Stage
    stage_col = None
    for c in merged.columns:
        if "stage" in str(c).lower() and "tumor" in str(c).lower():
            stage_col = c
            break
    if stage_col:
        merged["pathologic_stage"] = merged[stage_col]

    out_path = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"
    merged.to_csv(out_path, index=False)
    time.sleep(1) # Add a small delay to ensure file is written to disk
    print(f"Merged clinical + TNBC status saved to: {out_path}")

    # Add a check for file existence and size right after saving
    if out_path.exists():
        print(f"Confirmation: File {out_path} exists with size {os.path.getsize(out_path)} bytes.")
    else:
        print(f"Warning: File {out_path} was *not* found immediately after saving.")

    return merged


# __name__ == "__main__" block
# 1. GDC clinical
clin_gdc = get_gdc_clinical()

# 2. cBioPortal clinical (ER/PR/HER2)
clin_cbio = get_cbioportal_clinical("brca_tcga", "TCGA_BRCA")

# 3. Merge + define TNBC
merged = merge_and_define_TNBC()


GDC clinical saved to: tcga_tnbc_data/TCGA_BRCA_clinical_GDC.csv
Available clinical attributes for brca_tcga (ID: Display Name):
  AGE: Diagnosis Age
  AJCC_METASTASIS_PATHOLOGIC_PM: American Joint Committee on Cancer Metastasis Stage Code
  AJCC_NODES_PATHOLOGIC_PN: Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code
  AJCC_PATHOLOGIC_TUMOR_STAGE: Neoplasm Disease Stage American Joint Committee on Cancer Code
  AJCC_STAGING_EDITION: American Joint Committee on Cancer Publication Version Type
  AJCC_TUMOR_PATHOLOGIC_PT: American Joint Committee on Cancer Tumor Stage Code
  BRACHYTHERAPY_TOTAL_DOSE_POINT_A: Brachytherapy first reference point administered total dose
  CANCER_TYPE: Cancer Type
  CANCER_TYPE_DETAILED: Cancer Type Detailed
  CENT17_COPY_NUMBER: Cent17 Copy Number
  CLINICAL_STAGE: Neoplasm American Joint Committee on Cancer Clinical Group Stage
  CLIN_M_STAGE: Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage
  CLIN_N_STA

In [30]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")
clinical_cbio_path = OUTPUT_DIR / "TCGA_BRCA_clinical_cBio.csv"

if clinical_cbio_path.exists():
    df_cbio_check = pd.read_csv(clinical_cbio_path)
    print("Columns in TCGA_BRCA_clinical_cBio.csv:")
    print(df_cbio_check.columns.tolist())
else:
    print(f"File not found: {clinical_cbio_path}")

Columns in TCGA_BRCA_clinical_cBio.csv:
['patient_id', 'sample_id']


In [46]:
print(merged.columns.tolist())

['case_id', 'submitter_id', 'gender', 'race', 'ethnicity', 'age_at_diagnosis', 'tumor_stage', 'tumor_grade', 'primary_diagnosis', 'days_to_death', 'days_to_last_follow_up', 'vital_status', 'progression_or_recurrence', 'days_to_progression_or_recurrence', 'sample_type', 'project_id', 'Cancer Type', 'Cancer Type Detailed', 'Days to Sample Collection.', 'Fraction Genome Altered', 'Is FFPE', 'Mutation Count', 'Oct embedded', 'Oncotree Code', 'Other Sample ID', 'Pathology Report File Name', 'Pathology report uuid', 'Sample Initial Weight', 'Sample Type', 'Sample type id', 'Somatic Status', 'TMB (nonsynonymous)', 'Vial number', 'sample_id', 'TNBC', 'pathologic_stage']


**Output:**
`tcga_tnbc_data/TCGA_BRCA_clinical_TNBC.csv` with columns such as:

- `submitter_id`, `case_id`
- `age_at_diagnosis`, `gender`, `race`
- `pathologic_stage` (or similar)
- `vital_status`, `days_to_death`, `days_to_last_follow_up`
- `progression_or_recurrence`
- `ER_*`, `PR_*`, `HER2_*` (from cBioPortal)
- `TNBC` (1 = triple‑negative, 0 = other)
- `OS_months`, `OS_event` (if detected)

Use this file to filter TNBC samples: `TNBC == 1`.


In [7]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")

clinical_cbio_path = OUTPUT_DIR / "TCGA_BRCA_clinical_cBio.csv"
if clinical_cbio_path.exists():
    df_cbio = pd.read_csv(clinical_cbio_path)
    display(df_cbio.head(10))
else:
    print(f"File not found: {clinical_cbio_path}")

File not found: tcga_tnbc_data/TCGA_BRCA_clinical_cBio.csv


In [23]:
print(df_cbio.columns.tolist())

['patient_id', 'sample_id']


### 4) Cytoscape interaction table (from STRING)
This script:
- Reads your DEG list (you’ll create it in step 5, or use any gene list).
- Queries STRING for PPIs.
- Saves a TSV you can import into Cytoscape and then save as `.cys`.

File: `04_build_string_network.py`


In [6]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
from statsmodels.stats.multitest import multipletests

OUTPUT_DIR = Path("tcga_tnbc_data")


def simple_deg_analysis(
    rna_matrix_path,
    clinical_path,
    output_deg_path,
    tnbc_col="TNBC",
    min_samples_per_group=5,
):
    expr = pd.read_csv(rna_matrix_path, index_col=0)
    clin = pd.read_csv(clinical_path)

    # Map sample names in expression to submitter_id / case barcode
    # Expression columns are like: TCGA-XX-XXXX_ Primary Tumor
    # We'll extract the TCGA barcode part before the first underscore.
    def extract_barcode(col):
        return str(col).split("_")[0]

    sample_barcodes = [extract_barcode(c) for c in expr.columns]
    expr_tmp = expr.copy()
    expr_tmp.columns = sample_barcodes

    # Merge clinical info
    clin = clin.set_index("submitter_id")
    common = list(set(expr_tmp.columns) & set(clin.index))
    print(f"Common samples between expression and clinical: {len(common)}")

    expr_common = expr_tmp[common]
    clin_common = clin.loc[common]

    if tnbc_col not in clin_common.columns:
        raise ValueError(f"Column '{tnbc_col}' not found in clinical file.")

    group = clin_common[tnbc_col].astype(int)
    tnbc_samples = group[group == 1].index.tolist()
    other_samples = group[group == 0].index.tolist()

    print(f"Number of TNBC samples: {len(tnbc_samples)}")
    print(f"Number of non-TNBC samples: {len(other_samples)}")

    if len(tnbc_samples) < min_samples_per_group or len(other_samples) < min_samples_per_group:
        raise ValueError("Not enough samples in one of the groups.")

    # Simple t-test per gene
    genes = expr_common.index.tolist()
    logfc_list = []
    pvals = []

    for g in genes:
        x = expr_common.loc[g, tnbc_samples].astype(float).dropna()
        y = expr_common.loc[g, other_samples].astype(float).dropna()

        if len(x) < 2 or len(y) < 2:
            logfc_list.append(np.nan)
            pvals.append(1.0)
            continue

        m1 = np.log2(x + 1).mean()
        m2 = np.log2(y + 1).mean()
        logfc = m1 - m2

        # t-test on log2(x+1)
        # The original code has an error here, stats.t_ind_test does not exist.
        # It should be stats.ttest_ind
        t_stat, p_val = stats.ttest_ind(np.log2(x + 1), np.log2(y + 1), equal_var=False) # Assuming unequal variances
        logfc_list.append(logfc)
        pvals.append(p_val)

    deg_df = pd.DataFrame({
        "gene": genes,
        "log2FC": logfc_list,
        "pval": pvals,
    })

    # FDR correction
    _, adj_pvals, _, _ = multipletests(deg_df["pval"].fillna(1.0), method="fdr_bh")
    deg_df["adj_pval"] = adj_pvals

    # Sort by adj_pval
    deg_df = deg_df.sort_values("adj_pval").reset_index(drop=True)

    deg_df.to_csv(output_deg_path, index=False)
    print(f"DEG list saved to: {output_deg_path}")
    return deg_df


if __name__ == "__main__":
    rna_matrix = OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"
    clinical = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"
    output_deg = OUTPUT_DIR / "TNBC_DEG_list_for_GO_KEGG.csv"

    if not rna_matrix.exists():
        print("RNA-seq matrix not found. Run 01_download_rnaseq_matrix.py first.")
    elif not clinical.exists():
        print("Clinical file not found. Run 03_download_clinical_TNBC.py first.")
    else:
        deg = simple_deg_analysis(rna_matrix, clinical, output_deg)


RNA-seq matrix not found. Run 01_download_rnaseq_matrix.py first.


In [7]:
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")
rna_matrix_path = OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"
clinical_tnbc_path = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"

print(f"Checking for {rna_matrix_path}: {rna_matrix_path.exists()}")
print(f"Checking for {clinical_tnbc_path}: {clinical_tnbc_path.exists()}")

if rna_matrix_path.exists() and clinical_tnbc_path.exists():
    print("Both RNA-seq matrix and clinical TNBC files exist. Proceeding with DEG analysis.")
    # Re-run the DEG analysis if both files are confirmed to exist
    # (This logic is usually handled by the main DEG cell's __main__ block)
    # If the DEG cell consistently reports not found despite this check, there might be a path issue.
else:
    print("One or both required files are still missing. Please ensure previous steps completed successfully.")

Checking for tcga_tnbc_data/TCGA_BRCA_RNAseq_matrix.csv: False
Checking for tcga_tnbc_data/TCGA_BRCA_clinical_TNBC.csv: True
One or both required files are still missing. Please ensure previous steps completed successfully.


In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
from statsmodels.stats.multitest import multipletests

OUTPUT_DIR = Path("tcga_tnbc_data")


def simple_deg_analysis(
    rna_matrix_path,
    clinical_path,
    output_deg_path,
    tnbc_col="TNBC",
    min_samples_per_group=5,
):
    expr = pd.read_csv(rna_matrix_path, index_col=0)
    clin = pd.read_csv(clinical_path)

    # Map sample names in expression to submitter_id / case barcode
    # Expression columns are like: TCGA-XX-XXXX_ Primary Tumor
    # We'll extract the TCGA barcode part before the first underscore.
    def extract_barcode(col):
        return str(col).split("_")[0]

    sample_barcodes = [extract_barcode(c) for c in expr.columns]
    expr_tmp = expr.copy()
    expr_tmp.columns = sample_barcodes

    # Merge clinical info
    clin = clin.set_index("submitter_id")
    common = list(set(expr_tmp.columns) & set(clin.index))
    print(f"Common samples between expression and clinical: {len(common)}")

    expr_common = expr_tmp[common]
    clin_common = clin.loc[common]

    if tnbc_col not in clin_common.columns:
        raise ValueError(f"Column '{tnbc_col}' not found in clinical file.")

    group = clin_common[tnbc_col].astype(int)
    tnbc_samples = group[group == 1].index.tolist()
    other_samples = group[group == 0].index.tolist()

    print(f"Number of TNBC samples: {len(tnbc_samples)}")
    print(f"Number of non-TNBC samples: {len(other_samples)}")

    if len(tnbc_samples) < min_samples_per_group or len(other_samples) < min_samples_per_group:
        raise ValueError("Not enough samples in one of the groups.")

    # Simple t-test per gene
    genes = expr_common.index.tolist()
    logfc_list = []
    pvals = []

    for g in genes:
        x = expr_common.loc[g, tnbc_samples].astype(float).dropna()
        y = expr_common.loc[g, other_samples].astype(float).dropna()

        if len(x) < 2 or len(y) < 2:
            logfc_list.append(np.nan)
            pvals.append(1.0)
            continue

        m1 = np.log2(x + 1).mean()
        m2 = np.log2(y + 1).mean()
        logfc = m1 - m2

        # t-test on log2(x+1)
        # The original code has an error here, stats.t_ind_test does not exist.
        # It should be stats.ttest_ind
        t_stat, p_val = stats.ttest_ind(np.log2(x + 1), np.log2(y + 1), equal_var=False) # Assuming unequal variances
        logfc_list.append(logfc)
        pvals.append(p_val)

    deg_df = pd.DataFrame({
        "gene": genes,
        "log2FC": logfc_list,
        "pval": pvals,
    })

    # FDR correction
    _, adj_pvals, _, _ = multipletests(deg_df["pval"].fillna(1.0), method="fdr_bh")
    deg_df["adj_pval"] = adj_pvals

    # Sort by adj_pval
    deg_df = deg_df.sort_values("adj_pval").reset_index(drop=True)

    deg_df.to_csv(output_deg_path, index=False)
    print(f"DEG list saved to: {output_deg_path}")
    return deg_df


if __name__ == "__main__":
    rna_matrix = OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"
    clinical = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"
    output_deg = OUTPUT_DIR / "TNBC_DEG_list_for_GO_KEGG.csv"

    if not rna_matrix.exists():
        print("RNA-seq matrix not found. Run 01_download_rnaseq_matrix.py first.")
    elif not clinical.exists():
        print("Clinical file not found. Run 03_download_clinical_TNBC.py first.")
    else:
        deg = simple_deg_analysis(rna_matrix, clinical, output_deg)


RNA-seq matrix not found. Run 01_download_rnaseq_matrix.py first.


In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")
clinical_tnbc_path = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"
rna_matrix_path = OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"

if clinical_tnbc_path.exists() and rna_matrix_path.exists():
    clin = pd.read_csv(clinical_tnbc_path)
    expr = pd.read_csv(rna_matrix_path, index_col=0)

    # Map sample names in expression to submitter_id / case barcode
    def extract_barcode(col):
        return str(col).split("_")[0]
    sample_barcodes = [extract_barcode(c) for c in expr.columns]
    expr_tmp = expr.copy()
    expr_tmp.columns = sample_barcodes

    # Merge clinical info based on common submitter_ids
    clin_indexed = clin.set_index("submitter_id")
    common_samples = list(set(expr_tmp.columns) & set(clin_indexed.index))

    print(f"Total unique samples in RNA-seq matrix: {len(expr_tmp.columns)}")
    print(f"Total unique samples in clinical data: {len(clin_indexed.index)}")
    print(f"Common samples between RNA-seq and clinical data: {len(common_samples)}")

    if not common_samples:
        print("No common samples found between expression and clinical data.")
    else:
        clin_common = clin_indexed.loc[common_samples]
        print("\nDistribution of TNBC status in common samples:")
        print(clin_common["TNBC"].value_counts())

        tnbc_samples = clin_common[clin_common["TNBC"] == 1].index.tolist()
        other_samples = clin_common[clin_common["TNBC"] == 0].index.tolist()

        print(f"\nNumber of TNBC samples: {len(tnbc_samples)}")
        print(f"Number of non-TNBC samples: {len(other_samples)}")

        # Check for NaN values in TNBC column within common samples
        tnbc_nan_count = clin_common['TNBC'].isna().sum()
        if tnbc_nan_count > 0:
            print(f"Warning: {tnbc_nan_count} common samples have NaN for TNBC status and will be excluded.")

        if len(tnbc_samples) < 5:
            print(f"TNBC group has {len(tnbc_samples)} samples, which is less than the minimum required (5).")
        if len(other_samples) < 5:
            print(f"Non-TNBC group has {len(other_samples)} samples, which is less than the minimum required (5).")

else:
    print("Required files (TCGA_BRCA_RNAseq_matrix.csv or TCGA_BRCA_clinical_TNBC.csv) not found.")

In [1]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")
clinical_tnbc_path = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"

if clinical_tnbc_path.exists():
    clin_tnbc = pd.read_csv(clinical_tnbc_path)
    if 'TNBC' in clin_tnbc.columns:
        print("Value counts for 'TNBC' column:")
        print(clin_tnbc['TNBC'].value_counts())
    else:
        print("The 'TNBC' column was not found in the clinical DataFrame.")
else:
    print(f"Clinical file not found: {clinical_tnbc_path}")

Clinical file not found: tcga_tnbc_data/TCGA_BRCA_clinical_TNBC.csv


In [2]:
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")
clinical_tnbc_path = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"

if clinical_tnbc_path.exists():
    print(f"The file {clinical_tnbc_path} exists.")
else:
    print(f"The file {clinical_tnbc_path} does NOT exist.")
    print("Please ensure cell `9b49dc42` (which generates this file) has been run successfully.")

The file tcga_tnbc_data/TCGA_BRCA_clinical_TNBC.csv does NOT exist.
Please ensure cell `9b49dc42` (which generates this file) has been run successfully.


In [3]:
import requests
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")
STRING_API = "https://string-db.org/api"
ORGANISM = "Homo sapiens"
MIN_SCORE = 0.4  # combined score threshold


def fetch_string_interactions(gene_list, min_score=MIN_SCORE):
    """
    Query STRING for PPIs among a list of gene symbols.
    Returns edge DataFrame suitable for Cytoscape import.
    """
    # 1) Map gene symbols to STRING protein IDs
    identifiers = "\n".join(gene_list)
    params = {
        "identifiers": identifiers,
        "species": ORGANISM,
        "limit": 1,
        "echo_query": 1,
    }
    resp = requests.get(f"{STRING_API}/json/resolve_list", params=params)
    resp.raise_for_status()
    mapping = resp.json()

    sym2id = {}
    for entry in mapping:
        sym2id[entry["inputIdentifier"]] = entry["preferredName"]

    string_ids = list(set(sym2id.values()))
    if len(string_ids) < 2:
        print("Less than 2 genes mapped to STRING; no network.")
        return pd.DataFrame()

    # 2) Get interactions
    params2 = {
        "identifiers": "\n".join(string_ids),
        "species": ORGANISM,
        "add_whitecolor_nodes": "0",
        "limit": 1,
        "echo_query": 1,
    }
    resp2 = requests.get(f"{STRING_API}/json/get_interactions_list", params=params2)
    resp2.raise_for_status()
    interactions = resp2.json()

    edges = []
    for e in interactions:
        if e["score"] < min_score:
            continue
        edges.append({
            "protein1": e["protein1"],
            "protein2": e["protein2"],
            "combined_score": e["score"],
        })

    return pd.DataFrame(edges)


# __name__ == "__main__" block
# Input: DEG list from step 5 (or any gene list CSV with column 'gene')
deg_path = OUTPUT_DIR / "TNBC_DEG_list_for_GO_KEGG.csv"
if not deg_path.exists():
    print("DEG list not found. Run 05_prepare_deg_list.py first.")
else:
    deg = pd.read_csv(deg_path)
    # Expect a 'gene' column; adjust if needed
    gene_col = "gene" if "gene" in deg.columns else deg.columns[0]
    genes = deg[gene_col].dropna().astype(str).tolist()

    # Optionally take top N (e.g., 300)
    genes = genes[:300]

    edge_df = fetch_string_interactions(genes)
    if edge_df.empty:
        print("No interactions found.")
    else:
        out_path = OUTPUT_DIR / "STRING_TNBC_interactions.tsv"
        edge_df.to_csv(out_path, sep="\t", index=False)
        print(f"STRING interaction table saved to: {out_path}")
        print("Import this TSV into Cytoscape: File → Import → Network → File")


DEG list not found. Run 05_prepare_deg_list.py first.


### Download Validation Cohort: METABRIC TNBC Cases from cBioPortal

First, we need to identify the cBioPortal study ID for METABRIC breast cancer. Then we'll fetch its clinical data and define TNBC status for this cohort.

In [11]:
import requests
import pandas as pd
from pathlib import Path
import time

CBIO_API = "https://www.cbioportal.org/api"
OUTPUT_DIR = Path("tcga_tnbc_data")
OUTPUT_DIR.mkdir(exist_ok=True)

def list_cbioportal_studies(search_term="METABRIC"):
    url = f"{CBIO_API}/studies"
    params = {"pageSize": 1000} # Fetch a reasonable number of studies
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    studies = resp.json()

    print(f"Searching for studies containing '{search_term}'...")
    metabric_studies = [s for s in studies if search_term.lower() in s.get('name', '').lower() or search_term.lower() in s.get('description', '').lower() or search_term.lower() in s.get('studyId', '').lower()]

    if not metabric_studies:
        print(f"No studies found containing '{search_term}'.")
        return None

    print(f"Found {len(metabric_studies)} METABRIC-related studies:")
    for s in metabric_studies:
        print(f"  - Study ID: {s['studyId']}, Name: {s['name']}")

    # Heuristic: often the most relevant study is the one without a year or with a primary designation
    # For METABRIC, 'brca_metabric' is a common ID.
    for s in metabric_studies:
        if s['studyId'] == 'brca_metabric':
            return 'brca_metabric'

    # If 'brca_metabric' is not found, return the studyId of the first result as a fallback
    if metabric_studies:
        return metabric_studies[0]['studyId']
    return None

# Re-using the now-modified get_cbioportal_clinical function

def get_cbioportal_clinical(study_id, out_file_prefix="TCGA_BRCA"):
    """
    Fetch clinical data from cBioPortal (public API) for a given study.
    We'll get patient-level attributes including ER/PR/HER2 if available.
    """
    CBIO_API = "https://www.cbioportal.org/api"

    # 1. Get all patient IDs for the study
    url_patients = f"{CBIO_API}/studies/{study_id}/patients"
    params_patients = {"pageSize": 10000} # Get all patients
    resp_patients = requests.get(url_patients, params=params_patients)
    resp_patients.raise_for_status()
    patients_summary = resp_patients.json()
    patient_ids = [p['patientId'] for p in patients_summary]

    if not patient_ids:
        print(f"No patient data found for {study_id} from cBioPortal.")
        return pd.DataFrame()

    # 2. Get all clinical attribute metadata for the study
    url_clinical_attributes = f"{CBIO_API}/studies/{study_id}/clinical-attributes"
    params_clinical_attributes = {"pageSize": 10000}
    resp_clinical_attributes = requests.get(url_clinical_attributes, params=params_clinical_attributes)
    resp_clinical_attributes.raise_for_status()
    clinical_attribute_meta = resp_clinical_attributes.json()

    # Map clinical attribute IDs to display names for column headers
    attr_id_to_name = {attr['clinicalAttributeId']: attr['displayName'] for attr in clinical_attribute_meta}

    print(f"Available clinical attributes for {study_id} (ID: Display Name):")
    # For METABRIC, we specifically want to look for ER, PR, HER2 statuses
    # Let's list all attributes first to identify the correct ones.
    for attr in clinical_attribute_meta:
        print(f"  {attr['clinicalAttributeId']}: {attr['displayName']}")

    # It's important to identify the *correct* attribute IDs for ER/PR/HER2 for METABRIC.
    # Common ones are 'ER_STATUS', 'PR_STATUS', 'HER2_STATUS_BY_IHC', 'HER2_FISH_STATUS', 'HER2_STATUS'
    # We'll explicitly check for these or similar.
    possible_er_attr_ids = ["ER_STATUS", "ER_STATUS_BY_IMMUNOHISTOCHEMISTRY"]
    possible_pr_attr_ids = ["PR_STATUS", "PR_STATUS_BY_IMMUNOHISTOCHEMISTRY"]
    possible_her2_attr_ids = ["HER2_STATUS_BY_IHC", "HER2_STATUS", "HER2_FISH_STATUS"]

    metabric_relevant_attr_ids = []
    for attr_id_list in [possible_er_attr_ids, possible_pr_attr_ids, possible_her2_attr_ids]:
        for attr_id in attr_id_list:
            if attr_id in attr_id_to_name:
                metabric_relevant_attr_ids.append(attr_id)
                break # Found one, move to next receptor type

    if not metabric_relevant_attr_ids:
        print(f"Warning: No explicit ER/PR/HER2 status attributes found for {study_id} using common IDs. Fetching all available clinical attributes.")
        clinical_attribute_ids_str = ",".join(attr_id_to_name.keys())
    else:
        print(f"Detected receptor status attributes for {study_id}: {metabric_relevant_attr_ids}")
        clinical_attribute_ids_str = ",".join(metabric_relevant_attr_ids)

    # 3. Fetch clinical data for all patients and selected attributes using GET
    url_clinical_data = f"{CBIO_API}/studies/{study_id}/clinical-data"

    all_raw_clinical_data = []
    patient_id_chunks = [patient_ids[i:i + 100] for i in range(0, len(patient_ids), 100)] # Chunk patient IDs

    for i, patient_chunk in enumerate(patient_id_chunks):
        print(f"Fetching clinical data for patient chunk {i+1}/{len(patient_id_chunks)} for {study_id}...")
        # Prepare query parameters for GET request for the current chunk
        params_clinical_data = {
            "patientIds": ",".join(patient_chunk),
            "clinicalAttributeIds": clinical_attribute_ids_str # Only request relevant attributes
        }

        # Add retry logic for the GET request
        MAX_RETRIES = 5
        for attempt in range(MAX_RETRIES):
            try:
                resp_clinical_data = requests.get(url_clinical_data, params=params_clinical_data, timeout=60)
                resp_clinical_data.raise_for_status()
                all_raw_clinical_data.extend(resp_clinical_data.json())
                break # Break out of the retry loop if successful
            except requests.exceptions.RequestException as e:
                print(f"RequestException encountered for chunk {i+1} of {study_id} (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2 ** attempt) # Exponential back-off
                else:
                    raise # Re-raise the last exception if all retries fail
        else:
            # This block is executed if the loop completes without a 'break'
            print(f"Failed to retrieve clinical data for chunk {i+1} of {study_id} after {MAX_RETRIES} attempts. Skipping this chunk.")

    # DEBUG: Print unique clinicalAttributeIds found in the raw data
    if all_raw_clinical_data:
        raw_attribute_ids = {item.get('clinicalAttributeId') for item in all_raw_clinical_data}
        print(f"Unique clinicalAttributeIds found in raw data for {study_id} (before mapping/pivot):")
        print(list(raw_attribute_ids))
    else:
        print(f"No raw clinical data returned from cBioPortal for {study_id} for the requested attributes.")

    # 4. Process raw clinical data into a DataFrame
    # The data comes in a 'long' format, need to pivot it to 'wide'
    clinical_rows = []
    for item in all_raw_clinical_data:
        clinical_rows.append({
            "patient_id": item["patientId"],
            "attribute_name": attr_id_to_name.get(item["clinicalAttributeId"], item["clinicalAttributeId"]),
            "attribute_value": item["value"]
        })

    df_clinical_long = pd.DataFrame(clinical_rows)

    if df_clinical_long.empty:
        print(f"No detailed clinical attributes found for {study_id} from cBioPortal.")
        return pd.DataFrame()

    print(f"Unique attribute names found in raw clinical data for {study_id} (before pivot):")
    print(df_clinical_long['attribute_name'].unique().tolist())

    # Handle potential duplicate (patient_id, attribute_name) combinations
    # Keep the first occurrence if duplicates exist to allow pivoting.
    df_clinical_long = df_clinical_long.drop_duplicates(subset=['patient_id', 'attribute_name'], keep='first')

    # Pivot the table to get attributes as columns
    clin_cbio = df_clinical_long.pivot(index='patient_id', columns='attribute_name', values='attribute_value')
    clin_cbio = clin_cbio.reset_index()
    clin_cbio.columns.name = None # Remove the name of the columns index

    print(f"Columns found in cBioPortal clinical data for {study_id}:")
    print(clin_cbio.columns.tolist())

    # Add a dummy sample_id for merging purposes if not naturally present
    if 'sample_id' not in clin_cbio.columns:
        clin_cbio['sample_id'] = None # Or extract from patient_id if a pattern exists

    out_path_cbio = OUTPUT_DIR / f"{out_file_prefix}_clinical_cBio.csv"
    clin_cbio.to_csv(out_path_cbio, index=False)
    print(f"cBioPortal clinical for {study_id} saved to: {out_path_cbio}")
    return clin_cbio


metabric_study_id = list_cbioportal_studies("METABRIC")

if metabric_study_id:
    print(f"Identified primary METABRIC study ID: {metabric_study_id}")
    # Now call the (modified) get_cbioportal_clinical for METABRIC
    metabric_clinical_df = get_cbioportal_clinical(metabric_study_id, out_file_prefix="METABRIC")

    if not metabric_clinical_df.empty:
        print("METABRIC clinical data fetched successfully. Displaying head:")
        display(metabric_clinical_df.head())
    else:
        print("Failed to fetch METABRIC clinical data.")
else:
    print("Could not automatically identify a METABRIC study ID. Please manually check cBioPortal.")

Searching for studies containing 'METABRIC'...
Found 1 METABRIC-related studies:
  - Study ID: brca_metabric, Name: Breast Cancer (METABRIC, Nature 2012 & Nat Commun 2016)
Identified primary METABRIC study ID: brca_metabric
Available clinical attributes for brca_metabric (ID: Display Name):
  AGE_AT_DIAGNOSIS: Age at Diagnosis
  BREAST_SURGERY: Type of Breast Surgery
  CANCER_TYPE: Cancer Type
  CANCER_TYPE_DETAILED: Cancer Type Detailed
  CELLULARITY: Cellularity
  CHEMOTHERAPY: Chemotherapy
  CLAUDIN_SUBTYPE: Pam50 + Claudin-low subtype
  COHORT: Cohort
  ER_IHC: ER status measured by IHC
  ER_STATUS: ER Status
  GRADE: Neoplasm Histologic Grade
  HER2_SNP6: HER2 status measured by SNP6
  HER2_STATUS: HER2 Status
  HISTOLOGICAL_SUBTYPE: Tumor Other Histologic Subtype
  HORMONE_THERAPY: Hormone Therapy
  INFERRED_MENOPAUSAL_STATE: Inferred Menopausal State
  INTCLUST: Integrative Cluster
  LATERALITY: Primary Tumor Laterality
  LYMPH_NODES_EXAMINED_POSITIVE: Lymph nodes examined posit

,patient_id,Cancer Type,Cancer Type Detailed,ER Status,HER2 Status,Mutation Count,Neoplasm Histologic Grade,Oncotree Code,PR Status,Sample Type,TMB (nonsynonymous),Tumor Size,Tumor Stage,sample_id
0,MB-0000,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,NaN,3,IDC,Negative,Primary,0,22,2,None
1,MB-0002,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,2,3,IDC,Positive,Primary,2.615035408,10,1,None
2,MB-0005,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,2,2,IDC,Positive,Primary,2.615035408,15,2,None
3,MB-0006,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,1,2,MDLC,Positive,Primary,1.307517704,25,2,None
4,MB-0008,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,2,3,MDLC,Positive,Primary,2.615035408,40,2,None


### Define TNBC Status for METABRIC Cohort

Now that the METABRIC clinical data is downloaded, we will define the TNBC status based on ER, PR, and HER2 status, similar to the TCGA-BRCA cohort. The relevant columns found are `ER Status`, `PR Status`, and `HER2 Status`.

In [12]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")

def define_metabric_tnbc(metabric_clinical_df):
    df = metabric_clinical_df.copy()

    # Receptor status columns identified from previous step for METABRIC
    er_col = "ER Status"
    pr_col = "PR Status"
    her2_col = "HER2 Status"

    def is_negative_metabric(val):
        if pd.isna(val):
            return False
        v = str(val).strip().lower()
        # Common negative indicators for ER/PR/HER2 in METABRIC
        # Based on available clinical attributes from cBioPortal, values might be 'Negative', 'Positive'
        return v in ["negative", "0"]

    def is_positive_metabric(val):
        if pd.isna(val):
            return False
        v = str(val).strip().lower()
        return v in ["positive", "1"]

    if er_col in df.columns and pr_col in df.columns and her2_col in df.columns:
        df["ER_neg"] = df[er_col].apply(is_negative_metabric)
        df["PR_neg"] = df[pr_col].apply(is_negative_metabric)
        df["HER2_neg"] = df[her2_col].apply(is_negative_metabric)

        df["TNBC"] = (df["ER_neg"] & df["PR_neg"] & df["HER2_neg"]).astype(int)
        print(f"Defined TNBC status for METABRIC cohort. TNBC count: {df['TNBC'].sum()}")
    else:
        print("Could not detect all ER/PR/HER2 columns in METABRIC clinical data. TNBC column will not be created.")
        df["TNBC"] = pd.NA

    out_path = OUTPUT_DIR / "METABRIC_clinical_TNBC.csv"
    df.to_csv(out_path, index=False)
    print(f"METABRIC clinical data with TNBC status saved to: {out_path}")
    return df

# Call the function with the previously fetched METABRIC clinical DataFrame
if 'metabric_clinical_df' in locals() and not metabric_clinical_df.empty:
    metabric_clinical_tnbc_df = define_metabric_tnbc(metabric_clinical_df)
    if not metabric_clinical_tnbc_df.empty:
        print("METABRIC clinical data with TNBC status head:")
        display(metabric_clinical_tnbc_df.head())
else:
    print("METABRIC clinical data (metabric_clinical_df) not found or is empty. Please ensure previous step ran successfully.")

Defined TNBC status for METABRIC cohort. TNBC count: 320
METABRIC clinical data with TNBC status saved to: tcga_tnbc_data/METABRIC_clinical_TNBC.csv
METABRIC clinical data with TNBC status head:


,patient_id,Cancer Type,Cancer Type Detailed,ER Status,HER2 Status,Mutation Count,Neoplasm Histologic Grade,Oncotree Code,PR Status,Sample Type,TMB (nonsynonymous),Tumor Size,Tumor Stage,sample_id,ER_neg,PR_neg,HER2_neg,TNBC
0,MB-0000,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,NaN,3,IDC,Negative,Primary,0,22,2,None,False,True,True,0
1,MB-0002,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,2,3,IDC,Positive,Primary,2.615035408,10,1,None,False,False,True,0
2,MB-0005,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,2,2,IDC,Positive,Primary,2.615035408,15,2,None,False,False,True,0
3,MB-0006,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,1,2,MDLC,Positive,Primary,1.307517704,25,2,None,False,False,True,0
4,MB-0008,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,2,3,MDLC,Positive,Primary,2.615035408,40,2,None,False,False,True,0


In [13]:
import os
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")
RNASEQ_MATRIX_PATH = OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"

if RNASEQ_MATRIX_PATH.exists() and os.path.getsize(RNASEQ_MATRIX_PATH) > 0:
    print(f"Confirmation: The RNA-seq matrix '{RNASEQ_MATRIX_PATH}' exists and is not empty (size: {os.path.getsize(RNASEQ_MATRIX_PATH)} bytes).")
    # Optionally display head if it's not too big
    # import pandas as pd
    # df_rnaseq = pd.read_csv(RNASEQ_MATRIX_PATH, index_col=0)
    # print("RNA-seq matrix head:")
    # display(df_rnaseq.head())
else:
    print(f"Warning: The RNA-seq matrix '{RNASEQ_MATRIX_PATH}' is still missing or empty.")
    print("Please ensure the RNA-seq download and matrix building step (cell `9af4586e`) completed successfully.")


Please ensure the RNA-seq download and matrix building step (cell `9af4586e`) completed successfully.


**How to get .cys:**
1. Open Cytoscape.
2. File → Import → Network → File → select `STRING_TNBC_interactions.tsv`.
3. Set `protein1` and `protein2` as source/target.
4. After import, save as `your_network.cys` (File → Save).


In [ ]:
import os
from pathlib import Path

OUTPUT_DIR = Path("tcga_tnbc_data")

print(f"Contents of '{OUTPUT_DIR}':")
if OUTPUT_DIR.exists() and OUTPUT_DIR.is_dir():
    for item in sorted(os.listdir(OUTPUT_DIR)):
        item_path = OUTPUT_DIR / item
        if item_path.is_file():
            print(f"- {item} (File, Size: {item_path.stat().st_size} bytes)")
        elif item_path.is_dir():
            print(f"- {item} (Directory)")
        else:
            print(f"- {item} (Other)")
else:
    print(f"Directory '{OUTPUT_DIR}' does not exist.")

### 5) Gene list for GO/KEGG enrichment (DEG list)
This script:
- Loads your RNA‑seq matrix and clinical file.
- Filters TNBC vs non‑TNBC (using TNBC column from step 3).
- Performs a simple differential expression (t‑test per gene).
- Saves a gene list with log2FC, p‑value, adj‑p‑value.

For a real project, you’d use DESeq2/edgeR in R; this is a Python approximation.

File: `05_prepare_deg_list.py`


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
from statsmodels.stats.multitest import multipletests

OUTPUT_DIR = Path("tcga_tnbc_data")


def simple_deg_analysis(
    rna_matrix_path,
    clinical_path,
    output_deg_path,
    tnbc_col="TNBC",
    min_samples_per_group=5,
):
    expr = pd.read_csv(rna_matrix_path, index_col=0)
    clin = pd.read_csv(clinical_path)

    # Map sample names in expression to submitter_id / case barcode
    # Expression columns are like: TCGA-XX-XXXX_ Primary Tumor
    # We'll extract the TCGA barcode part before the first underscore.
    def extract_barcode(col):
        return str(col).split("_")[0]

    sample_barcodes = [extract_barcode(c) for c in expr.columns]
    expr_tmp = expr.copy()
    expr_tmp.columns = sample_barcodes

    # Merge clinical info
    clin = clin.set_index("submitter_id")
    common = list(set(expr_tmp.columns) & set(clin.index))
    print(f"Common samples between expression and clinical: {len(common)}")

    expr_common = expr_tmp[common]
    clin_common = clin.loc[common]

    if tnbc_col not in clin_common.columns:
        raise ValueError(f"Column '{tnbc_col}' not found in clinical file.")

    group = clin_common[tnbc_col].astype(int)
    tnbc_samples = group[group == 1].index.tolist()
    other_samples = group[group == 0].index.tolist()

    if len(tnbc_samples) < min_samples_per_group or len(other_samples) < min_samples_per_group:
        raise ValueError("Not enough samples in one of the groups.")

    # Simple t-test per gene
    genes = expr_common.index.tolist()
    logfc_list = []
    pvals = []

    for g in genes:
        x = expr_common.loc[g, tnbc_samples].astype(float).dropna()
        y = expr_common.loc[g, other_samples].astype(float).dropna()

        if len(x) < 2 or len(y) < 2:
            logfc_list.append(np.nan)
            pvals.append(1.0)
            continue

        m1 = np.log2(x + 1).mean()
        m2 = np.log2(y + 1).mean()
        logfc = m1 - m2

        # t-test on log2(x+1)
        # The original code has an error here, stats.t_ind_test does not exist.
        # It should be stats.ttest_ind
        t_stat, p_val = stats.ttest_ind(np.log2(x + 1), np.log2(y + 1), equal_var=False) # Assuming unequal variances
        logfc_list.append(logfc)
        pvals.append(p_val)

    deg_df = pd.DataFrame({
        "gene": genes,
        "log2FC": logfc_list,
        "pval": pvals,
    })

    # FDR correction
    _, adj_pvals, _, _ = multipletests(deg_df["pval"].fillna(1.0), method="fdr_bh")
    deg_df["adj_pval"] = adj_pvals

    # Sort by adj_pval
    deg_df = deg_df.sort_values("adj_pval").reset_index(drop=True)

    deg_df.to_csv(output_deg_path, index=False)
    print(f"DEG list saved to: {output_deg_path}")
    return deg_df


# __name__ == "__main__" block
rna_matrix = OUTPUT_DIR / "TCGA_BRCA_RNAseq_matrix.csv"
clinical = OUTPUT_DIR / "TCGA_BRCA_clinical_TNBC.csv"
output_deg = OUTPUT_DIR / "TNBC_DEG_list_for_GO_KEGG.csv"

if not rna_matrix.exists():
    print("RNA-seq matrix not found. Run 01_download_rnaseq_matrix.py first.")
elif not clinical.exists():
    print("Clinical file not found. Run 03_download_clinical_TNBC.py first.")
else:
    deg = simple_deg_analysis(rna_matrix, clinical, output_deg)


RNA-seq matrix not found. Run 01_download_rnaseq_matrix.py first.


**Output:**
`tcga_tnbc_data/TNBC_DEG_list_for_GO_KEGG.csv` with:
- `gene`
- `log2FC` (TNBC vs non‑TNBC)
- `pval`
- `adj_pval`

Use this file for:
- g:Profiler, Enrichr, DAVID, or
- R clusterProfiler for GO/KEGG.


### How to run everything in order
In your terminal (or Colab cell with `!` prefix):


In [1]:
# To run these scripts in order, execute the following commands in separate cells:
# !python 01_download_rnaseq_matrix.py
# !python 02_download_mirna_matrix.py
# !python 03_download_clinical_TNBC.py
# !python 05_prepare_deg_list.py
# !python 04_build_string_network.py
# Note: In Colab, you would typically run the individual Python cells I generated above,
# rather than calling them as separate Python scripts from bash.
# I've commented out the bash commands as they are generally for terminal execution.